# **1. Import Library**
Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import joblib

# **2. Memuat Dataset dari Hasil Clustering**
Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [7]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("data_clustering_inverse.csv")

Saving data_clustering_inverse.csv to data_clustering_inverse (1).csv


In [8]:
df.head()

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,AgeGroup,Target
0,14.09,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,Tinggi,1
1,376.24,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,Tinggi,0
2,126.29,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,Rendah,1
3,184.50,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,Rendah,1
4,92.15,Debit,Oklahoma City,ATM,18.0,Student,172.0,1.0,781.68,Rendah,1


# **Feature Encoding: One Hot Encoding**

In [9]:
categorical_cols = list(df.select_dtypes(include=['object']).columns)


df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)


df_encoded.head()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,Target,TransactionType_Debit,Location_Atlanta,Location_Austin,Location_Baltimore,...,Location_Tucson,Location_Virginia Beach,Location_Washington,Channel_Branch,Channel_Online,CustomerOccupation_Engineer,CustomerOccupation_Retired,CustomerOccupation_Student,AgeGroup_Sedang,AgeGroup_Tinggi
0,14.09,70.0,81.0,1.0,5112.21,1,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,376.24,68.0,141.0,1.0,13758.91,0,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,126.29,19.0,56.0,1.0,1122.35,1,True,False,False,False,...,False,False,False,False,True,False,False,True,False,False
3,184.50,26.0,25.0,1.0,8569.06,1,True,False,False,False,...,False,False,False,False,True,False,False,True,False,False
4,92.15,18.0,172.0,1.0,781.68,1,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False


# **3. Data Splitting**
Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [10]:
X = df_encoded.drop('Target', axis=1)


y = df_encoded['Target']
X_train, X_test, y_train, y_test = train_test_split(
      X,
      y,
      test_size = 0.3,
      random_state = 42,
      stratify = y
)




print("Jumlah data total: ",len(X))
print("Jumlah data latih: ",len(X_train))
print("Jumlah data test: ",len(X_test))

Jumlah data total:  1945
Jumlah data latih:  1361
Jumlah data test:  584


# **4. Membangun Model Klasifikasi**

In [11]:
decision_tree_model = DecisionTreeClassifier(random_state=42)

decision_tree_model.fit(X_train, y_train)


DecisionTreeClassifier(random_state=42)

In [12]:
joblib.dump(decision_tree_model, 'decision_tree_model.h5')

['decision_tree_model.h5']

# **Melatih model menggunakan algoritma klasifikasi scikit-learn dengan model RandomForestClassifier**

In [13]:
new_model = RandomForestClassifier(random_state=42)

new_model.fit(X_train, y_train)

y_pred_dt = decision_tree_model.predict(X_test)
y_pred_new = new_model.predict(X_test)


print("Decision Tree Performance") #classification_report untuk Decision Tree
print(classification_report(y_test, y_pred_dt))

print("="*50)

print("New Model Performance") #classification_report untuk New Model
print(classification_report(y_test, y_pred_new))


Decision Tree Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       294
           1       1.00      1.00      1.00       290

    accuracy                           1.00       584
   macro avg       1.00      1.00      1.00       584
weighted avg       1.00      1.00      1.00       584

New Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       294
           1       1.00      1.00      1.00       290

    accuracy                           1.00       584
   macro avg       1.00      1.00      1.00       584
weighted avg       1.00      1.00      1.00       584



In [14]:
joblib.dump(new_model, 'explore_random_forest_classification.h5')

['explore_random_forest_classification.h5']

# **Hyperparameter Tuning Model**



In [15]:
params = {
          'n_estimators': [100, 200,300],
          'max_depth': [10, 20, 30],
          'min_samples_split': [2, 5, 10],
          'min_samples_leaf': [1, 2, 4],
          'max_features': ['sqrt', 'log2']
         }

new_model_tuned = GridSearchCV(
     estimator = RandomForestClassifier(random_state=42),
     param_grid = params,
     cv = 5,
     scoring = 'accuracy'
)


new_model_tuned.fit(X_train, y_train)


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [10, 20, 30],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [16]:
y_pred_tuning = new_model_tuned.predict(X_test)

print("Tuned Model Performance")
print(classification_report(y_test, y_pred_tuning))

Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       294
           1       1.00      1.00      1.00       290

    accuracy                           1.00       584
   macro avg       1.00      1.00      1.00       584
weighted avg       1.00      1.00      1.00       584



In [17]:
joblib.dump(new_model_tuned, 'tuning_classification.h5')

['tuning_classification.h5']

End of Code